# Bulk Query Export

Runs the queries defined in `queries/queries.xml` and writes their raw results to `raw_exports/<appliance-name>/`.

## Setup

Select an appliance from `config.yaml` and configure export behaviour.

In [ ]:
import re
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display
from tideway import notebooks as tw_nb

APPLIANCE_NAME = None
APPLIANCE_INDEX = 0
INCLUDE_QUERY_TITLES = None  # Exact titles; None exports every query
EXCLUDE_QUERY_TITLES = []
OVERWRITE_EXISTING = False
RAW_EXPORT_BASE_DIR = None  # Defaults to <repo>/raw_exports
QUERY_PAGE_SIZE = 100
QUERY_RECORD_LIMIT = None
DISPLAY_ROW_LIMIT = 50

display(HTML('''<style>
.jp-OutputArea-output, .output_area, .dataframe { width: 100% !important; }
.dataframe { table-layout: auto !important; }
</style>'''))

repo_root = tw_nb.find_repo_root(Path.cwd())
config = tw_nb.load_config()
selected_config = tw_nb.selected_appliance_config(config, APPLIANCE_NAME, APPLIANCE_INDEX)
appliance_name = str(selected_config.get('name') or selected_config.get('target') or 'default').strip()
tw = tw_nb.appliance_from_config(appliance_name=APPLIANCE_NAME, appliance_index=APPLIANCE_INDEX)
query_file = repo_root / 'queries' / 'queries.xml'
if RAW_EXPORT_BASE_DIR:
    configured_export_root = Path(RAW_EXPORT_BASE_DIR).expanduser()
    raw_export_root = configured_export_root if configured_export_root.is_absolute() else repo_root / configured_export_root
else:
    raw_export_root = repo_root / 'raw_exports'
raw_export_root = raw_export_root.resolve()

def slugify(value):
    return re.sub(r'[^A-Za-z0-9]+', '_', str(value)).strip('_').lower() or 'unnamed'

export_dir = raw_export_root / slugify(appliance_name)
export_dir.mkdir(parents=True, exist_ok=True)

print('Repository root:', repo_root)
print('Appliance name :', appliance_name)
print('Target         :', tw.target)
print('Query file     :', query_file)
print('Export folder  :', export_dir)

## Load Queries

In [ ]:
def load_query_definitions(path):
    if not path.exists():
        raise FileNotFoundError(f'Query file not found: {path}')

    root = ET.parse(path).getroot()
    definitions = []
    for element in root.findall('query'):
        title = (element.get('title') or 'Untitled Query').strip()
        search = (element.findtext('search') or '').strip()
        if search:
            definitions.append({'title': title, 'search': search})
    return definitions

query_definitions = load_query_definitions(query_file)
include_titles = set(INCLUDE_QUERY_TITLES or [])
exclude_titles = set(EXCLUDE_QUERY_TITLES or [])
selected_queries = [
    query for query in query_definitions
    if (not include_titles or query['title'] in include_titles)
    and query['title'] not in exclude_titles
]

unknown_titles = include_titles - {query['title'] for query in query_definitions}
if unknown_titles:
    raise ValueError(f'Unknown query titles: {sorted(unknown_titles)}')
if not selected_queries:
    raise ValueError('No queries selected for export.')

print(f'Loaded {len(query_definitions)} queries; selected {len(selected_queries)}.')

## Export Queries

In [ ]:
search_api = tw.data()
execution_summary = []

for position, query in enumerate(selected_queries, start=1):
    title = query['title']
    output_path = export_dir / f"{slugify(title)}.csv"
    print(f'[{position}/{len(selected_queries)}] {title}')

    if output_path.exists() and not OVERWRITE_EXISTING:
        print(f'  Skipped existing file: {output_path.name}')
        execution_summary.append({
            'Query': title, 'Rows': None, 'Status': 'skipped_existing', 'Path': str(output_path)
        })
        continue

    try:
        rows = search_api.search(
            {'query': query['search']},
            format='object',
            limit=QUERY_PAGE_SIZE,
            record_limit=QUERY_RECORD_LIMIT,
        )
        if hasattr(rows, 'ok'):
            status = getattr(rows, 'status_code', 'unknown')
            detail = getattr(rows, 'text', '')
            raise RuntimeError(f'API request failed ({status}): {detail[:500]}')

        frame = pd.DataFrame(rows or [])
        frame.insert(0, 'Appliance Target', tw.target)
        frame.insert(1, 'Appliance Name', appliance_name)
        frame.insert(2, 'Query Title', title)
        frame.to_csv(output_path, index=False)
        print(f'  Saved {len(frame)} rows to {output_path.name}')
        execution_summary.append({
            'Query': title, 'Rows': len(frame), 'Status': 'ok', 'Path': str(output_path)
        })
    except Exception as exc:
        print(f'  Failed: {exc}')
        execution_summary.append({
            'Query': title, 'Rows': 0, 'Status': f'error: {exc}', 'Path': str(output_path)
        })

summary_df = pd.DataFrame(execution_summary)
display(summary_df.head(DISPLAY_ROW_LIMIT).style.set_table_styles([
    {'selector': 'table', 'props': [('width', '100%')]}
]))
print('Completed:', int((summary_df['Status'] == 'ok').sum()))
print('Skipped  :', int((summary_df['Status'] == 'skipped_existing').sum()))
print('Failed   :', int(summary_df['Status'].str.startswith('error:').sum()))